# Energieverbrauchsanalyse: Auswirkung eines neuen Geräts

## 1. Einleitung und Datengrundlage

### Datensatz
- **Quelle**: CSV-Dateien mit 15-Minuten-Intervallen für Energieverbrauchsdaten.
- **Zeitraum**: April 24 – Juni 1, 2026.
- **Installationsdatum**: 13. Mai 2026.

### Spalten in den CSV-Dateien
- **Gesamtverbrauch / Mittelwerte [W]**: Gesamtenergieverbrauch (Summe aller Quellen).
- **Netzbezug / Mittelwerte [W]**: Energiebezug aus dem Netz.
- **Batterieentladung / Mittelwerte [W]**: Energie aus der Batterie.
- **Direktverbrauch / Mittelwerte [W]**: Direkter Verbrauch (z. B. von PV-Anlage).

### Ziel
Nachweis der Auswirkung des neuen Geräts (10–20W) auf den nächtlichen Energieverbrauch.

## 2. Vorbereitung: Bibliotheken und Konfiguration

Hier laden wir die benötigten Bibliotheken und definieren die wichtigsten Parameter für die Analyse.

In [ ]:
# Import required libraries
import os
import glob
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
from scipy import stats

# Configuration
DATA_DIR = 'energy_data'
NIGHTTIME_START = 0  # Start of nighttime window (00:00)
CONSUMPTION_LOWER = 80  # Lower bound for normal consumption (W)
CONSUMPTION_UPPER = 130  # Upper bound for normal consumption (W)
INSTALLATION_DATE = datetime(2026, 5, 13)

## 3. Datenladeroutine

Hier definieren wir Funktionen, um die CSV-Dateien zu parsen und die relevanten Daten zu extrahieren.

In [ ]:
def parse_time(time_str):
    # Parse time string like '01:00' to hour and minute
    time_str = time_str.strip('"=').strip()
    try:
        hour, minute = map(int, time_str.split(':'))
        return hour, minute
    except:
        return None, None

def get_date_from_filename(filename):
    # Extract date from filename like Energiebilanz_2026_05_12.csv
    basename = os.path.basename(filename)
    date_str = basename.replace('Energiebilanz_', '').replace('.csv', '')
    try:
        return datetime.strptime(date_str, '%Y_%m_%d')
    except:
        return None

def process_file(filepath):
    # Process a single CSV file and return smart nighttime data
    date = get_date_from_filename(filepath)
    if date is None:
        print(f'Skipping {filepath} (invalid date)')
        return None
    
    try:
        df = pd.read_csv(filepath, delimiter=';', skipinitialspace=True, encoding='utf-8-sig')
    except:
        print(f'Failed to read {filepath}')
        return None
    
    time_col = df.iloc[:, 0]
    gesamtverbrauch_col = df.iloc[:, 4]
    
    intervals = []
    nighttime_end_hour = None
    
    for idx, time_str in enumerate(time_col):
        hour, minute = parse_time(time_str)
        if hour is None:
            continue
        
        gesamtverbrauch = gesamtverbrauch_col.iloc[idx]
        if pd.isna(gesamtverbrauch):
            continue
        
        intervals.append({
            'date': date,
            'hour': hour,
            'minute': minute,
            'Gesamtverbrauch': gesamtverbrauch
        })
        
        if hour >= NIGHTTIME_START and gesamtverbrauch > CONSUMPTION_UPPER:
            if nighttime_end_hour is None:
                nighttime_end_hour = hour
    
    if nighttime_end_hour is None:
        nighttime_end_hour = 5
    
    nighttime_data = []
    for interval in intervals:
        hour = interval['hour']
        gesamtverbrauch = interval['Gesamtverbrauch']
        
        if (NIGHTTIME_START <= hour < nighttime_end_hour and 
            CONSUMPTION_LOWER <= gesamtverbrauch <= CONSUMPTION_UPPER):
            nighttime_data.append(interval)
    
    if not nighttime_data:
        print(f'No smart nighttime data for {date.strftime("%Y-%m-%d")}')
    
    return pd.DataFrame(nighttime_data)

## 4. Daten laden und vorbereiten

In [ ]:
csv_files = glob.glob(os.path.join(DATA_DIR, 'Energiebilanz_*.csv'))
print(f'Found {len(csv_files)} CSV files in {DATA_DIR}/')

all_nighttime_data = []
for filepath in csv_files:
    df = process_file(filepath)
    if df is not None and not df.empty:
        all_nighttime_data.append(df)

if not all_nighttime_data:
    print('No smart nighttime data found!')
else:
    nighttime_df = pd.concat(all_nighttime_data, ignore_index=True)
    print(f'Total smart nighttime intervals: {len(nighttime_df)}')

## 5. Datensegmentierung: Vor und nach der Installation

In [ ]:
pre_installation = nighttime_df[nighttime_df['date'] < INSTALLATION_DATE]
post_installation = nighttime_df[nighttime_df['date'] >= INSTALLATION_DATE]

print(f'Pre-installation (before {INSTALLATION_DATE.strftime("%Y-%m-%d")}):')
print(f'  - Days: {pre_installation["date"].nunique()}')
print(f'  - Intervals: {len(pre_installation)}')

print(f'Post-installation (on/after {INSTALLATION_DATE.strftime("%Y-%m-%d")}):')
print(f'  - Days: {post_installation["date"].nunique()}')
print(f'  - Intervals: {len(post_installation)}')

## 6. Statistische Analyse

### A. Deskriptive Statistik
Hier berechnen wir den durchschnittlichen Energieverbrauch für beide Gruppen.

### B. t-Test
Ein t-Test (Welch's t-test) wird durchgeführt, um zu prüfen, ob der Unterschied zwischen den beiden Gruppen statistisch signifikant ist.
- **Nullhypothese (H0)**: Es gibt keinen Unterschied im durchschnittlichen Verbrauch.
- **Alternativhypothese (H1)**: Es gibt einen Unterschied.
- **Signifikanzniveau**: 5% (p < 0.05).

In [ ]:
pre_avg = pre_installation['Gesamtverbrauch'].mean()
pre_std = pre_installation['Gesamtverbrauch'].std()
post_avg = post_installation['Gesamtverbrauch'].mean()
post_std = post_installation['Gesamtverbrauch'].std()
delta = post_avg - pre_avg

print(f'Pre-installation avg Gesamtverbrauch: {pre_avg:.2f} W (±{pre_std:.2f} W)')
print(f'Post-installation avg Gesamtverbrauch: {post_avg:.2f} W (±{post_std:.2f} W)')
print(f'Delta (Post - Pre): {delta:.2f} W')

t_stat, p_value = stats.ttest_ind(
    post_installation['Gesamtverbrauch'],
    pre_installation['Gesamtverbrauch'],
    equal_var=False
)

print(f'Welch t-test: t-statistic={t_stat:.3f}, p-value={p_value:.4f}')
print(f'Significant (p < 0.05): {"Yes" if p_value < 0.05 else "No"}')

## 7. Visualisierung

In [ ]:
plt.figure(figsize=(12, 6))

daily_pre = pre_installation.groupby('date')['Gesamtverbrauch'].mean()
daily_post = post_installation.groupby('date')['Gesamtverbrauch'].mean()

plt.scatter(daily_pre.index, daily_pre.values, color='blue', label='Pre-installation')
plt.scatter(daily_post.index, daily_post.values, color='red', label='Post-installation')

plt.axhline(y=pre_avg, color='blue', linestyle='--', label=f'Pre-avg: {pre_avg:.1f} W')
plt.axhline(y=post_avg, color='red', linestyle='--', label=f'Post-avg: {post_avg:.1f} W')

plt.title('Nighttime Gesamtverbrauch (00:00 - Dynamic End, 80W-130W)')
plt.xlabel('Date')
plt.ylabel('Gesamtverbrauch [W]')
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Interpretation der Ergebnisse

### Schlussfolgerung
- **Kein signifikanter Unterschied**: Der p-Wert ist größer als 0.05.
- **Mögliche Gründe**: Gerät <5W, nicht nachts aktiv, separater Stromkreis, oder 15-Minuten-Mittelung.
- **Empfehlung**: Mehr Daten sammeln oder andere Spalten prüfen.

## 9. Abhängigkeiten

Installation: `pip install pandas numpy scipy matplotlib`